<a href="https://colab.research.google.com/github/thunwa2524/nxp_k64F_temperature_measure/blob/main/analyzetemp_uart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import make_interp_spline

# ชื่อไฟล์ Log จาก PuTTY
file_name = 'putty_data.txt'

try:
    # 1. อ่านไฟล์และจัดการข้อมูล
    df = pd.read_csv(file_name, header=None, names=['Temperature'])
    df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')
    df = df.dropna().reset_index(drop=True)

    if len(df) < 4:
        print("⚠️ ข้อมูลน้อยเกินไปสำหรับการทำเส้นโค้ง (ต้องการอย่างน้อย 4 จุด)")
    else:
        x = df.index
        y = df['Temperature']

        # คำนวณค่าสถิติ Max, Min และ Average
        temp_max = y.max()
        temp_min = y.min()
        temp_avg = y.mean()
        idx_max = y.idxmax()
        idx_min = y.idxmin()

        print("=" * 35)
        print(f"📊 สรุปผลข้อมูลอุณหภูมิ ({len(df)} จุด)")
        print(f"🔺 สูงสุด (Max)     : {temp_max:.2f} °C (จุดที่ {idx_max})")
        print(f"🔻 ต่ำสุด (Min)     : {temp_min:.2f} °C (จุดที่ {idx_min})")
        print(f"➗ ค่าเฉลี่ย (Avg)  : {temp_avg:.2f} °C")
        print("=" * 35)

        # 2. สร้างเส้นโค้งสมูท Cubic Spline
        x_smooth = np.linspace(x.min(), x.max(), 300)
        spline = make_interp_spline(x, y, k=3)
        y_smooth = spline(x_smooth)

        # 3. ตั้งค่ากราฟ
        fig, ax = plt.subplots(figsize=(12, 6.5))

        ax.plot(
            x_smooth,
            y_smooth,
            color='#000080',
            linewidth=2.5,
            label='Trend Line'
        )

        ax.fill_between(
            x_smooth,
            y_smooth,
            y_smooth.min() - 0.5,
            color='#000080',
            alpha=0.12
        )

        # ax.scatter(
        #     x,
        #     y,
        #     color='#DC143C',
        #     s=15,
        #     zorder=20,
        #     label='Actual Data'
        # )

        # เส้นค่าเฉลี่ย และจุด Max/Min
        ax.axhline(
            temp_avg,
            color='#2E8B57',
            linestyle='--',
            linewidth=1.8,
            label=f'Avg: {temp_avg:.2f} °C'
        )

        ax.scatter(
            idx_max,
            temp_max,
            color='#FF4500',
            s=100,
            marker='^',
            zorder=25,
            label=f'Max: {temp_max:.2f} °C'
        )

        ax.scatter(
            idx_min,
            temp_min,
            color='#1E90FF',
            s=100,
            marker='v',
            zorder=25,
            label=f'Min: {temp_min:.2f} °C'
        )

        # ย้าย Legend ไปด้านนอกกราฟด้านบน
        ax.legend(
            loc='lower center',
            bbox_to_anchor=(0.5, 1.02),
            ncol=5,
            frameon=True,
            facecolor='#F8F9FA',
            edgecolor='#DDDDDD',
            fontsize=10
        )

        ax.set_title(
            'Temperature Trend from UART',
            fontsize=16,
            fontweight='bold',
            pad=42
        )
        ax.set_xlabel('Data Point (Sample)', fontsize=12)
        ax.set_ylabel('Temperature (°C)', fontsize=12)

        # เพิ่มระยะว่างด้านบนและด้านล่างกราฟ
        y_range = y_smooth.max() - y_smooth.min()
        margin = max(y_range * 0.25, 0.5)

        ax.set_ylim(
            bottom=y_smooth.min() - (margin * 0.5),
            top=y_smooth.max() + margin
        )

        ax.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()

except Exception as e:
    print("Error:", e)